# Storytelling, Stakeholder Communication and Writing an Analysis Memo

Notebook for **Chapter 14** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch14.html

Tip: change a number, predict the output, then run the cell again.

## 14.1 Ask the Right Questions: Stakeholder Management

## 14.2 Data Storytelling: Headline, Evidence, Recommendation

In [ ]:
import pandas as pd
r=pd.DataFrame({"city":["Mumbai","Delhi","Bengaluru","Pune"],"jan":[40000,30000,20000,10000],"feb":[41000,20000,19000,10000]})
r["chg"]=r.feb-r.jan; r["pct"]=(r.chg/r.jan*100).round(1)
print(r.to_string(index=False)); print(r[["jan","feb","chg"]].sum().to_dict())

## 14.3 Writing: Executive Summaries, Emails and the Analysis Memo

## 14.4 Presenting and Handling Questions

## 14.5 Reproducibility and Documentation

In [ ]:
import pandas as pd
d=pd.DataFrame({"city":["Delhi"]*3+["Pune"],"delay_flag":[1,0,0,1],"order_value":[450.0,None,520.5,300.0],"order_date":pd.to_datetime(["2025-02-01"]*4)})
dd=pd.DataFrame({"column":d.columns,"dtype":d.dtypes.astype(str).values,"nulls":d.isna().sum().values,"example":[d[c].dropna().iloc[0] for c in d.columns]})
print(dd.to_string(index=False))

## 14.6 Why Two Dashboards Disagree: Reconciliation, Definitions, Contracts and Git

In [ ]:
import pandas as pd
orders = pd.DataFrame({
 "order_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 11],
 "ts_utc": ["2025-03-01 10:00", "2025-03-05 12:00", "2025-03-10 08:00", "2025-03-15 18:00", "2025-03-20 09:00", "2025-03-28 20:00",
            "2025-03-31 19:00", "2025-03-31 20:30", "2025-04-01 01:00", "2025-03-12 11:00", "2025-03-12 11:00", "2025-03-25 14:00"],
 "amount": [1000, 2000, 1500, 2500, 1800, 3000, 2200, 1300, 900, 700, 700, 1600],
 "status": ["delivered", "delivered", "delivered", "delivered", "returned", "delivered",
            "delivered", "delivered", "delivered", "delivered", "delivered", "cancelled"]})
orders["ts_utc"] = pd.to_datetime(orders["ts_utc"])
orders["ts_ist"] = orders["ts_utc"] + pd.Timedelta(hours=5, minutes=30)      # India Standard Time is UTC + 5:30
def in_march(col):
    return (orders[col] >= "2025-03-01") & (orders[col] < "2025-04-01")

a = orders[in_march("ts_utc")].amount.sum()
d1 = orders[in_march("ts_utc")]
dup = d1[d1.duplicated("order_id")].amount.sum()                 # same order loaded twice
d2 = d1.drop_duplicates("order_id")
not_delivered = d2[d2.status != "delivered"].amount.sum()        # returned or cancelled
d3 = d2[d2.status == "delivered"]
tz = d3[~in_march("ts_ist").reindex(d3.index)].amount.sum()      # March in UTC but April in IST
b = orders.drop_duplicates("order_id")
b = b[(b.status == "delivered") & in_march("ts_ist").reindex(b.index)].amount.sum()
bridge = pd.DataFrame({"step": ["Marketing (UTC, all rows)", "duplicate row: order 10 loaded twice",
                                "returned or cancelled", "timezone: orders 7 and 8", "Finance (IST, delivered, unique)"],
                       "amount": [a, -dup, -not_delivered, -tz, b]})
print(bridge.to_string(index=False))
print("pieces add up:", a - dup - not_delivered - tz == b)

                                step  amount
           Marketing (UTC, all rows)   18300
duplicate row: order 10 loaded twice    -700
               returned or cancelled   -3400
            timezone: orders 7 and 8   -3500
    Finance (IST, delivered, unique)   10700
pieces add up: True


In [ ]:
import pandas as pd
orders = pd.DataFrame({
 "order_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 11],
 "ts_utc": ["2025-03-01 10:00", "2025-03-05 12:00", "2025-03-10 08:00", "2025-03-15 18:00", "2025-03-20 09:00", "2025-03-28 20:00",
            "2025-03-31 19:00", "2025-03-31 20:30", "2025-04-01 01:00", "2025-03-12 11:00", "2025-03-12 11:00", "2025-03-25 14:00"],
 "amount": [1000, 2000, 1500, 2500, 1800, 3000, 2200, 1300, 900, 700, 700, 1600],
 "status": ["delivered", "delivered", "delivered", "delivered", "returned", "delivered",
            "delivered", "delivered", "delivered", "delivered", "delivered", "cancelled"]})
orders["ts_utc"] = pd.to_datetime(orders["ts_utc"])
orders["ts_ist"] = orders["ts_utc"] + pd.Timedelta(hours=5, minutes=30)      # India Standard Time is UTC + 5:30
def in_march(col):
    return (orders[col] >= "2025-03-01") & (orders[col] < "2025-04-01")
import sqlite3
con = sqlite3.connect(":memory:")
orders.assign(ts_utc=orders.ts_utc.astype(str)).drop(columns="ts_ist").to_sql("orders", con, index=False)

METRICS = {
    "net_revenue": {
        "owner": "Finance",
        "definition": "Sum of amount of delivered orders, one row per order, by Indian-time order date.",
        "sql": """SELECT SUM(amount) FROM (
                      SELECT DISTINCT order_id, amount, status,
                             DATE(ts_utc, '+330 minutes') AS d FROM orders)
                  WHERE status = 'delivered' AND d >= :start AND d < :end"""},
    "gross_revenue": {
        "owner": "Marketing",
        "definition": "Sum of amount of all unique orders regardless of status, by Indian-time order date.",
        "sql": """SELECT SUM(amount) FROM (
                      SELECT DISTINCT order_id, amount, DATE(ts_utc, '+330 minutes') AS d FROM orders)
                  WHERE d >= :start AND d < :end"""},
}

def value(name, start, end):
    return con.execute(METRICS[name]["sql"], {"start": start, "end": end}).fetchone()[0]

for name in METRICS:
    print(name, value(name, "2025-03-01", "2025-04-01"), "|", METRICS[name]["owner"])

net_revenue 10700 | Finance
gross_revenue 14100 | Marketing


In [ ]:
import pandas as pd
orders = pd.DataFrame({
 "order_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 11],
 "ts_utc": ["2025-03-01 10:00", "2025-03-05 12:00", "2025-03-10 08:00", "2025-03-15 18:00", "2025-03-20 09:00", "2025-03-28 20:00",
            "2025-03-31 19:00", "2025-03-31 20:30", "2025-04-01 01:00", "2025-03-12 11:00", "2025-03-12 11:00", "2025-03-25 14:00"],
 "amount": [1000, 2000, 1500, 2500, 1800, 3000, 2200, 1300, 900, 700, 700, 1600],
 "status": ["delivered", "delivered", "delivered", "delivered", "returned", "delivered",
            "delivered", "delivered", "delivered", "delivered", "delivered", "cancelled"]})
orders["ts_utc"] = pd.to_datetime(orders["ts_utc"])
orders["ts_ist"] = orders["ts_utc"] + pd.Timedelta(hours=5, minutes=30)      # India Standard Time is UTC + 5:30
def in_march(col):
    return (orders[col] >= "2025-03-01") & (orders[col] < "2025-04-01")
checks = {
    "order_id is unique":        lambda d: d["order_id"].is_unique,
    "amount is positive":        lambda d: bool((d["amount"] > 0).all()),
    "status in allowed list":    lambda d: d["status"].isin(["delivered", "returned", "cancelled"]).all(),
    "no missing timestamps":     lambda d: d["ts_utc"].notna().all(),
    "no future timestamps":      lambda d: bool((d["ts_utc"] <= pd.Timestamp("2025-04-30")).all()),
}
def run_checks(d):
    failed = [name for name, test in checks.items() if not test(d)]
    for name in checks:
        print(("FAIL " if name in failed else "ok   ") + name)
    return failed

print("raw table:")
failed = run_checks(orders)
if failed:
    print("-> stop the load and alert the owner:", failed)
print("after deduplicating:")
run_checks(orders.drop_duplicates("order_id"))

raw table:
FAIL order_id is unique
ok   amount is positive
ok   status in allowed list
ok   no missing timestamps
ok   no future timestamps
-> stop the load and alert the owner: ['order_id is unique']
after deduplicating:
ok   order_id is unique
ok   amount is positive
ok   status in allowed list
ok   no missing timestamps
ok   no future timestamps


## Chapter recap

## Practice set

Problem 1 (easy). Rewrite the chart title "Monthly orders" as an action title. Orders were 250 in January and 240 in February. **Answer:** "Orders fell 4% in February (250 to 240)". Check: (240 − 250) / 250 = −4%.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch14.html